Сгенерировать последовательности, которые состоят из цифр (от 0 до 9) и задаются следующим образом:<br>
x - последовательность цифр<br>
**y1 = x1**<br>
**yi = xi + x1**<br>
Если yi >= 10 то yi = yi - 10<br>
Научить модель рекуррентной нейронной сети предсказывать yi по xi. Использовать: RNN, LSTM, GRU

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

Генерируем датасет из случайных последовательностей согласно функции:

In [6]:
PAD_TOKEN = 10  # Индекс для обозначения пустоты
VOCAB_SIZE = 11

def generate_variable_sequences(num_samples=1000):
    X_raw = []
    Y_raw = []
    
    for _ in range(num_samples):
        # Генерируем случайную длину (например, от 10 до 50)
        seq_len = np.random.randint(10, 51)
        
        x_seq = np.random.randint(0, 10, size=seq_len)
        y_seq = np.zeros(seq_len, dtype=int)
        
        y_seq[0] = x_seq[0]
        for i in range(1, seq_len):
            val = x_seq[i] + x_seq[0]
            if val >= 10: val -= 10
            y_seq[i] = val
            
        X_raw.append(torch.LongTensor(x_seq))
        Y_raw.append(torch.LongTensor(y_seq))
        
    return X_raw, Y_raw

X_raw, Y_raw = generate_variable_sequences()

Заполняем пустоты в последовательностях для выравнивания батчей:

In [7]:
import torch.nn.utils.rnn as rnn_utils

# Упаковываем в батч, добавляем токен для пустот PAD_TOKEN
X_padded = rnn_utils.pad_sequence(X_raw, batch_first=True, padding_value=PAD_TOKEN)
Y_padded = rnn_utils.pad_sequence(Y_raw, batch_first=True, padding_value=PAD_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Создаем класс нейронной сети:

In [12]:
vocab_size = VOCAB_SIZE

class UniversalRNN(nn.Module):
    def __init__(self, cell_type, vocab_size, embedding_dim=30, hidden_dim=128):
        super(UniversalRNN, self).__init__()
        # Слой эмбеддингов: переводит индексы в векторы
        self.embedding = nn.Embedding(vocab_size, embedding_dim) 

        if cell_type == 'RNN':
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x) 
        
        # Для Many-to-Many берем первый выход, h_n игнорируем
        out, _ = self.rnn(embedded)
        # out: (batch_size, seq_length, hidden_dim)
        
        out = self.fc(out) 
        # out: (batch_size, seq_length, vocab_size)
        return out

Тренируем и сравниваем нейронные сети с ячейками RNN, LSTM и GRU

In [15]:
import time

def train_and_eval(cell_type, epochs=300):
    model = UniversalRNN(vocab_size=VOCAB_SIZE, cell_type=cell_type).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN) # Игнорируем пустоты при вычислении лосса
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    start_time = time.time()
    model.train()
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_padded.to(device))
        
        loss = criterion(outputs.view(-1, VOCAB_SIZE), Y_padded.to(device).view(-1))
        loss.backward()
        optimizer.step()
    
    total_time = time.time() - start_time
    
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X_padded.to(device)), dim=2)
        mask = (Y_padded != PAD_TOKEN)
        correct = ((preds.cpu() == Y_padded) & mask).sum().item()
        total = mask.sum().item()
        accuracy = correct / total
        
    return loss.item(), accuracy, total_time


results = {}
for name in ['RNN', 'LSTM', 'GRU']:
    print(f"Тренировка {name}...")
    results[name] = train_and_eval(name)


print("\n| Модель | Final Loss | Accuracy | Time (s) |")
print("|-------|------------|----------|----------|")
for name, (loss, acc, t) in results.items():
    print(f"| {name:5} | {loss:.6f} | {acc:.2%} | {t:.2f}s |")

Тренировка RNN...
Тренировка LSTM...
Тренировка GRU...

| Модель | Final Loss | Accuracy | Time (s) |
|-------|------------|----------|----------|
| RNN   | 1.612356 | 36.33% | 0.32s |
| LSTM  | 0.002624 | 100.00% | 0.81s |
| GRU   | 0.002653 | 100.00% | 0.64s |
